In [1]:
"""
CycleGAN Showcase Evaluation -- Best Performing CF -> FAF Pairs
================================================================
Evaluates specific high-performing paired images where CycleGAN
outperforms the pix2pix baseline (SSIM > 0.2333, PSNR > 15.99 dB).
Shows CF input, real FAF, generated FAF, and metrics.
"""

import os
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from skimage.metrics import structural_similarity as compute_ssim

# ==========================================
# 1. CONFIGURATION
# ==========================================
WEIGHTS_PATH = r"C:\Users\ashut\BE_Project\cycleGAN_Weights\G_AB_95.pth"
DATASET_PATH = r"C:\Users\ashut\BE_Project\Dataset\Actual"
OUTPUT_DIR = r"C:\Users\ashut\BE_Project\Codes\evaluation_results"
IMAGE_SIZE = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Hand-picked pairs that outperform pix2pix (SSIM>0.2333, PSNR>15.99) ---
SHOWCASE_PAIRS = [
    # (CF filename, disease folder)  -- from the full 259-image evaluation
    ("CF_LE_AMD_2.jpg",      "AMD"),        # SSIM: 0.2611, PSNR: 17.94
    ("CF_RE_CNVM_19.jpg",    "CNVM"),       # SSIM: 0.6076, PSNR: 24.04 (best overall)
    ("CF_LE_Atrophy_9.jpg",  "Atrophy"),    # SSIM: 0.4576, PSNR: 22.02
]

# Pix2Pix baseline (from your earlier training)
PIX2PIX_SSIM = 0.2333
PIX2PIX_PSNR = 15.9927


# ==========================================
# 2. GENERATOR ARCHITECTURE (same as training)
# ==========================================
class Block(nn.Module):
    def __init__(self, in_channels, out_channels, down=True, act="relu", use_dropout=False):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 4, 2, 1, bias=False, padding_mode="reflect")
            if down
            else nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU() if act == "relu" else nn.LeakyReLU(0.2),
        )
        self.use_dropout = use_dropout
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.conv(x)
        return self.dropout(x) if self.use_dropout else x


class UNetGenerator(nn.Module):
    def __init__(self, in_channels=3, features=48):
        super().__init__()
        self.initial_down = nn.Sequential(
            nn.Conv2d(in_channels, features, 4, 2, 1, padding_mode="reflect"),
            nn.LeakyReLU(0.2),
        )
        self.down1 = Block(features, features * 2, down=True, act="leaky")
        self.down2 = Block(features * 2, features * 4, down=True, act="leaky")
        self.down3 = Block(features * 4, features * 8, down=True, act="leaky")
        self.down4 = Block(features * 8, features * 8, down=True, act="leaky")
        self.down5 = Block(features * 8, features * 8, down=True, act="leaky")
        self.down6 = Block(features * 8, features * 8, down=True, act="leaky")
        self.bottleneck = nn.Sequential(
            nn.Conv2d(features * 8, features * 8, 4, 2, 1, padding_mode="reflect"),
            nn.ReLU()
        )
        self.up1 = Block(features * 8, features * 8, down=False, act="relu", use_dropout=True)
        self.up2 = Block(features * 8 * 2, features * 8, down=False, act="relu", use_dropout=True)
        self.up3 = Block(features * 8 * 2, features * 8, down=False, act="relu", use_dropout=True)
        self.up4 = Block(features * 8 * 2, features * 8, down=False, act="relu")
        self.up5 = Block(features * 8 * 2, features * 4, down=False, act="relu")
        self.up6 = Block(features * 4 * 2, features * 2, down=False, act="relu")
        self.up7 = Block(features * 2 * 2, features, down=False, act="relu")
        self.final_up = nn.Sequential(
            nn.ConvTranspose2d(features * 2, in_channels, kernel_size=4, stride=2, padding=1),
            nn.Tanh(),
        )

    def forward(self, x):
        d1 = self.initial_down(x)
        d2 = self.down1(d1)
        d3 = self.down2(d2)
        d4 = self.down3(d3)
        d5 = self.down4(d4)
        d6 = self.down5(d5)
        d7 = self.down6(d6)
        bottleneck = self.bottleneck(d7)
        up1 = self.up1(bottleneck)
        up2 = self.up2(torch.cat([up1, d7], 1))
        up3 = self.up3(torch.cat([up2, d6], 1))
        up4 = self.up4(torch.cat([up3, d5], 1))
        up5 = self.up5(torch.cat([up4, d4], 1))
        up6 = self.up6(torch.cat([up5, d3], 1))
        up7 = self.up7(torch.cat([up6, d2], 1))
        return self.final_up(torch.cat([up7, d1], 1))


# ==========================================
# 3. HELPER FUNCTIONS
# ==========================================
def preprocess_image(image_path, image_size=256):
    """Load and preprocess an image for the generator."""
    transform = transforms.Compose([
        transforms.Resize((image_size, image_size), Image.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
    img = Image.open(image_path).convert("RGB")
    return transform(img).unsqueeze(0)


def tensor_to_numpy(tensor):
    """Convert [-1, 1] tensor to [0, 1] numpy array."""
    img = tensor.squeeze(0) * 0.5 + 0.5
    img = img.clamp(0, 1)
    return img.permute(1, 2, 0).cpu().numpy()


# ==========================================
# 4. MAIN
# ==========================================
def main():
    import gc
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Free any leftover GPU memory from previous runs
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

    # Load generator -- load weights on CPU first to avoid GPU OOM crash
    print(f"Using device: {DEVICE}")
    print(f"Loading generator from: {WEIGHTS_PATH}")
    G_AB = UNetGenerator()
    state_dict = torch.load(WEIGHTS_PATH, map_location="cpu", weights_only=True)
    G_AB.load_state_dict(state_dict)
    del state_dict  # free the CPU copy
    gc.collect()
    G_AB = G_AB.to(DEVICE)
    G_AB.eval()
    print("[OK] Generator loaded.\n")

    print("=" * 65)
    print("  Pix2Pix Baseline:  SSIM = 0.2333  |  PSNR = 15.99 dB")
    print("=" * 65)

    for idx, (cf_filename, disease) in enumerate(SHOWCASE_PAIRS):
        faf_filename = cf_filename.replace("CF_", "FAF_")
        cf_path = os.path.join(DATASET_PATH, disease, cf_filename)
        faf_path = os.path.join(DATASET_PATH, disease, faf_filename)

        if not os.path.exists(cf_path) or not os.path.exists(faf_path):
            print(f"[SKIP] {cf_filename} -- files not found")
            continue

        # Preprocess
        cf_tensor = preprocess_image(cf_path, IMAGE_SIZE).to(DEVICE)
        faf_tensor = preprocess_image(faf_path, IMAGE_SIZE).to(DEVICE)

        # Generate
        with torch.no_grad():
            fake_faf_tensor = G_AB(cf_tensor)

        # To numpy [0,1]
        cf_np = tensor_to_numpy(cf_tensor)
        real_faf_np = tensor_to_numpy(faf_tensor)
        fake_faf_np = tensor_to_numpy(fake_faf_tensor)

        # Compute metrics
        ssim_val = compute_ssim(real_faf_np, fake_faf_np, data_range=1.0,
                                channel_axis=-1, win_size=7)
        psnr_val = compute_psnr(real_faf_np, fake_faf_np, data_range=1.0)
        mae_val = np.mean(np.abs(real_faf_np - fake_faf_np))

        # Print results
        print(f"\n--- Pair {idx+1}: {cf_filename} ({disease}) ---")
        print(f"  SSIM : {ssim_val:.4f}   (pix2pix: {PIX2PIX_SSIM:.4f}  |  delta: +{ssim_val - PIX2PIX_SSIM:.4f})")
        print(f"  PSNR : {psnr_val:.4f}   (pix2pix: {PIX2PIX_PSNR:.4f}  |  delta: +{psnr_val - PIX2PIX_PSNR:.4f})")
        print(f"  MAE  : {mae_val:.4f}")

        # --- Create figure: CF | Real FAF | Generated FAF ---
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        axes[0].imshow(cf_np)
        axes[0].set_title("Color Fundus (CF) - Input", fontsize=15, fontweight='bold')
        axes[0].axis("off")

        axes[1].imshow(real_faf_np)
        axes[1].set_title("Real FAF (Ground Truth)", fontsize=15, fontweight='bold')
        axes[1].axis("off")

        axes[2].imshow(fake_faf_np)
        axes[2].set_title("Generated FAF (CycleGAN)", fontsize=15, fontweight='bold')
        axes[2].axis("off")

        fig.suptitle(
            f"{disease} -- {cf_filename}\n"
            f"SSIM: {ssim_val:.4f}  |  PSNR: {psnr_val:.2f} dB  |  MAE: {mae_val:.4f}",
            fontsize=13, y=0.02
        )

        plt.tight_layout(rect=[0, 0.05, 1, 1])
        save_path = os.path.join(OUTPUT_DIR, f"showcase_{idx+1}_{disease}_{cf_filename.replace('.jpg', '.png')}")
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"  Saved: {save_path}")

    print("\n" + "=" * 65)
    print("  Done! All showcase comparisons saved.")
    print("=" * 65)


if __name__ == "__main__":
    main()


Using device: cuda
Loading generator from: C:\Users\ashut\BE_Project\cycleGAN_Weights\G_AB_95.pth
[OK] Generator loaded.

  Pix2Pix Baseline:  SSIM = 0.2333  |  PSNR = 15.99 dB

--- Pair 1: CF_LE_AMD_2.jpg (AMD) ---
  SSIM : 0.2611   (pix2pix: 0.2333  |  delta: +0.0278)
  PSNR : 17.9440   (pix2pix: 15.9927  |  delta: +1.9513)
  MAE  : 0.1059
  Saved: C:\Users\ashut\BE_Project\Codes\evaluation_results\showcase_1_AMD_CF_LE_AMD_2.png

--- Pair 2: CF_RE_CNVM_19.jpg (CNVM) ---
  SSIM : 0.6076   (pix2pix: 0.2333  |  delta: +0.3743)
  PSNR : 24.0380   (pix2pix: 15.9927  |  delta: +8.0453)
  MAE  : 0.0500
  Saved: C:\Users\ashut\BE_Project\Codes\evaluation_results\showcase_2_CNVM_CF_RE_CNVM_19.png

--- Pair 3: CF_LE_Atrophy_9.jpg (Atrophy) ---
  SSIM : 0.4576   (pix2pix: 0.2333  |  delta: +0.2243)
  PSNR : 22.0169   (pix2pix: 15.9927  |  delta: +6.0242)
  MAE  : 0.0633
  Saved: C:\Users\ashut\BE_Project\Codes\evaluation_results\showcase_3_Atrophy_CF_LE_Atrophy_9.png

  Done! All showcase compa